In [ ]:
import os
import sys

try:
    import amanzi_xml
except ImportError:
    amanzi_xml_path = os.path.join(os.environ["AMANZI_SRC_DIR"], "tools","amanzi_xml")
    sys.path.append(amanzi_xml_path)
    import amanzi_xml

from amanzi_xml.utils import search as asearch
from amanzi_xml.utils import io as aio
from amanzi_xml.utils import errors as aerrors
from amanzi_xml.common import parameter, parameter_list

from scipy.io import loadmat
import numpy as np

from pathlib import Path

In [ ]:
# Parameters cell
import config_utils
config = config_utils.load_config('config.json')
watershed_name = config['case']['watershed_name']
hucs           = [config['case']['hucs']]
site_name      = config['case']['site_name']
meshsize_nx    = config['case']['meshsize_nx']

# # simulation control
# start_year_spinup         = config['start_year_spinup']
# end_year_spinup           = config['end_year_spinup']
# nyears_steadystate_spinup = config['nyears_steadystate_spinup']
# nyears_cyclic_spinup      = config['nyears_cyclic_spinup']
# start_year_transient      = config['start_year_transient']
# end_year_transient        = config['end_year_transient']

flag_scenario = 's1' # choose which scenario
outputs = {}

In [ ]:
m2_mat_filename =  f'../data-processed/{site_name}/m2_coords_{site_name}.mat'
loaded_data = loadmat(m2_mat_filename)
dzs_soil  = loaded_data['dzs_soil'].flatten()
dzs_geo   = loaded_data['dzs_geo'].flatten()
meshsize_nz = len(dzs_soil) + len(dzs_geo)

In [ ]:
base_dir = Path().resolve().parent

tmp_xml_run2 = base_dir / f'caselambda-run2-prefire.{flag_scenario}' / f'{site_name}_nx{meshsize_nx}_nz{meshsize_nz}.run2.v1.6_pflotran.xml'

# Load hillslope shape

In [ ]:
# settings copied from "1-full_workflow_OakCreek.ipynb"
# site_name = 'NF01'
# meshsize_nx is loaded from config.json above.

#dzs_soil = [0.05, 0.05, 0.05, 0.1, 0.25, 0.5, 0.5, 0.5]
#dzs_geo = [1., 1., 1.5, 1.5, 2., 2., 2., 3., 3., 3.]
#left, right, topl, bottoml, topr, bottomr = 0.0, 680.0, 1102.011, 1080.011, 972.046, 950.046
#front, back   = -0.5, 0.5

m2_mat_filename =  f'../data-processed/{site_name}/m2_coords_{site_name}.mat'
loaded_data = loadmat(m2_mat_filename)
dzs_soil  = loaded_data['dzs_soil'].flatten()
dzs_geo   = loaded_data['dzs_geo'].flatten()
m2_coords = loaded_data['m2_coords']

In [ ]:
#print(m2_coords)
total_thickness = np.sum(dzs_soil) + np.sum(dzs_geo)
left, right   = m2_coords[0,0], m2_coords[meshsize_nx,0]
topl, bottoml = m2_coords[0,2], m2_coords[0,2] - total_thickness
topr, bottomr = m2_coords[meshsize_nx,2], m2_coords[meshsize_nx,2] - total_thickness
front, back   = m2_coords[0,1], m2_coords[meshsize_nx+1,1]

print(left, right, topl, bottoml, topr, bottomr)
print(front, back)

# use points to define boundary cells and faces "region"

In [ ]:
# list of 3D points
dzs_merge = np.concatenate((dzs_soil, dzs_geo), axis=0)

points_bd_face_left_subsurface = [] # boundary face - left subsurface domain

for i in range(len(dzs_merge)):
    x = left
    y = 0.0
    # The sum should be up to i-1, so if i is 0, the sum is 0
    sum_dzs_merge_prev = sum(dzs_merge[0:i])
    z = topl - sum_dzs_merge_prev - (dzs_merge[i] / 2)
    points_bd_face_left_subsurface.append([x, y, z])

print(points_bd_face_left_subsurface)


points_bd_face_right_subsurface = [] # boundary face - right subsurface domain

for i in range(len(dzs_merge)):
    x = right
    y = 0.0
    # The sum should be up to i-1, so if i is 0, the sum is 0
    sum_dzs_merge_prev = sum(dzs_merge[0:i])
    z = topr - sum_dzs_merge_prev - (dzs_merge[i] / 2)
    points_bd_face_right_subsurface.append([x, y, z])

print(points_bd_face_right_subsurface)

In [ ]:
points_bd_cell_left_subsurface = [] # boundary face - left subsurface domain

for i in range(len(dzs_merge)):
    x = (left + m2_coords[1,0])/2
    y = 0.0
    # The sum should be up to i-1, so if i is 0, the sum is 0
    sum_dzs_merge_prev = sum(dzs_merge[0:i])
    z = (topl + m2_coords[1,2])/2 - sum_dzs_merge_prev - (dzs_merge[i] / 2)
    points_bd_cell_left_subsurface.append([x, y, z])

print(points_bd_cell_left_subsurface)


points_bd_cell_right_subsurface = [] # boundary face - right subsurface domain

for i in range(len(dzs_merge)):
    x = (right + m2_coords[-2,0])/2
    y = 0.0
    # The sum should be up to i-1, so if i is 0, the sum is 0
    sum_dzs_merge_prev = sum(dzs_merge[0:i])
    z = (topr + m2_coords[-2,2])/2 - sum_dzs_merge_prev - (dzs_merge[i] / 2)
    points_bd_cell_right_subsurface.append([x, y, z])

print(points_bd_cell_right_subsurface)

In [ ]:
points_bd_face_left_surface = []

x = (left + m2_coords[1,0])/2
y = 0.0
z = (topl + m2_coords[1,2])/2

points_bd_face_left_surface.append([x, y, z])


points_bd_face_right_surface = []

x = (right + m2_coords[-2,0])/2
y = 0.0
z = (topr + m2_coords[-2,2])/2

points_bd_face_right_surface.append([x, y, z])

# Load xml and add observations for boundary mass fluxes

In [ ]:
def add_observations_to_xml(fin_xml, points_bd_face_left_subsurface, points_bd_face_right_subsurface,
                            points_bd_cell_left_subsurface, points_bd_cell_right_subsurface,
                            points_bd_face_left_surface, points_bd_face_right_surface):
    fin_xml = str(fin_xml)
    xml = aio.fromFile(fin_xml, True)

    regions = asearch.find_path(xml, ["regions",], no_skip=True)
    for i, point in enumerate(points_bd_face_left_subsurface):
        regions.sublist(f"obs region bdface left subsurface {i}").sublist("region: point") \
                                                  .setParameter("coordinate", "Array(double)", point)
    for i, point in enumerate(points_bd_face_right_subsurface):
        regions.sublist(f"obs region bdface right subsurface {i}").sublist("region: point") \
                                                  .setParameter("coordinate", "Array(double)", point)
    for i, point in enumerate(points_bd_cell_left_subsurface):
        regions.sublist(f"obs region bdcell left subsurface {i}").sublist("region: point") \
                                                  .setParameter("coordinate", "Array(double)", point)
    for i, point in enumerate(points_bd_cell_right_subsurface):
        regions.sublist(f"obs region bdcell right subsurface {i}").sublist("region: point") \
                                                  .setParameter("coordinate", "Array(double)", point)

    regions.sublist(f"obs region bdface left surface").sublist("region: point") \
                                                .setParameter("coordinate", "Array(double)", points_bd_face_left_surface[0])
    regions.sublist(f"obs region bdface right surface").sublist("region: point") \
                                                .setParameter("coordinate", "Array(double)", points_bd_face_right_surface[0])

    obs = asearch.find_path(xml, ["observations",], no_skip=True)

    for var in ['water_flux']:
        var_obs = obs.sublist(var+"_observation")
        var_obs.setParameter("observation output filename", "string", var+".dat")
        var_obs.setParameter("times start period stop", "Array(double)", [0,1,-1])
        var_obs.setParameter("times start period stop units", "string", "d")
        var_obs.setParameter("time units", "string", "d")
        columns = var_obs.sublist("observed quantities")
        for i, point in enumerate(points_bd_face_left_subsurface):
            region_name = f"obs region bdface left subsurface {i}"
            obs_reg = columns.sublist(region_name)
            obs_reg.setParameter("variable", "string", var)
            obs_reg.setParameter("direction normalized flux", "bool", "true")
            obs_reg.setParameter("region", "string", region_name)
            obs_reg.setParameter("functional", "string", "extensive integral")
            obs_reg.setParameter("location name", "string", "face")
            obs_reg.setParameter("time integrated", "bool", "true")
        for i, point in enumerate(points_bd_face_right_subsurface):
            region_name = f"obs region bdface right subsurface {i}"
            obs_reg = columns.sublist(region_name)
            obs_reg.setParameter("variable", "string", var)
            obs_reg.setParameter("direction normalized flux", "bool", "true")
            obs_reg.setParameter("region", "string", region_name)
            obs_reg.setParameter("functional", "string", "extensive integral")
            obs_reg.setParameter("location name", "string", "face")
            obs_reg.setParameter("time integrated", "bool", "true")

    for var in ['total_component_concentration']:
        var_obs = obs.sublist(var+"_observation")
        var_obs.setParameter("observation output filename", "string", var+".dat")
        var_obs.setParameter("times start period stop", "Array(double)", [0,1,-1])
        var_obs.setParameter("times start period stop units", "string", "d")
        var_obs.setParameter("time units", "string", "d")
        columns = var_obs.sublist("observed quantities")
        for i, point in enumerate(points_bd_cell_left_subsurface):
            region_name = f"obs region bdcell left subsurface {i}"
            obs_reg = columns.sublist(region_name)
            obs_reg.setParameter("variable", "string", var)
            obs_reg.setParameter("region", "string", region_name)
            obs_reg.setParameter("functional", "string", "average")
            obs_reg.setParameter("location name", "string", "cell")
            obs_reg.setParameter("time integrated", "bool", "false")
        for i, point in enumerate(points_bd_cell_right_subsurface):
            region_name = f"obs region bdcell right subsurface {i}"
            obs_reg = columns.sublist(region_name)
            obs_reg.setParameter("variable", "string", var)
            obs_reg.setParameter("region", "string", region_name)
            obs_reg.setParameter("functional", "string", "average")
            obs_reg.setParameter("location name", "string", "cell")
            obs_reg.setParameter("time integrated", "bool", "false")

    for var in ['surface-total_component_concentration']:
        var_obs = obs.sublist(var+"_observation")
        var_obs.setParameter("observation output filename", "string", var+".dat")
        var_obs.setParameter("times start period stop", "Array(double)", [0,1,-1])
        var_obs.setParameter("times start period stop units", "string", "d")
        var_obs.setParameter("time units", "string", "d")
        columns = var_obs.sublist("observed quantities")
        region_name = "obs region bdface left surface"
        obs_reg = columns.sublist(region_name)
        obs_reg.setParameter("variable", "string", var)
        obs_reg.setParameter("region", "string", region_name)
        obs_reg.setParameter("functional", "string", "average")
        obs_reg.setParameter("location name", "string", "cell")
        obs_reg.setParameter("time integrated", "bool", "false")
        region_name = "obs region bdface right surface"
        obs_reg = columns.sublist(region_name)
        obs_reg.setParameter("variable", "string", var)
        obs_reg.setParameter("region", "string", region_name)
        obs_reg.setParameter("functional", "string", "average")
        obs_reg.setParameter("location name", "string", "cell")
        obs_reg.setParameter("time integrated", "bool", "false")

    fout_xml = fin_xml.split('.xml')[0]+'.obs4massflux.xml'
    aio.toFile(xml, fout_xml)

# Apply to Lambda Run2 only
add_observations_to_xml(
    tmp_xml_run2,
    points_bd_face_left_subsurface,
    points_bd_face_right_subsurface,
    points_bd_cell_left_subsurface,
    points_bd_cell_right_subsurface,
    points_bd_face_left_surface,
    points_bd_face_right_surface
)
